# 02 · Train, evaluate, compare, export (GPU session)

Attach: (1) the same private MC-MED dataset, (2) the **output of notebook 01**. Cached data stages are read
from notebook 01's output, so nothing is preprocessed again.

Training checkpoints every epoch (`last.pt`). If the session is cut off, rerun the same cell: it resumes.

In [ ]:
# --- Setup: install this repo (code only; data comes from the attached Kaggle dataset) ---
# Private repo? Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets).
import os, subprocess, sys
REPO = "github.com/<YOUR_GITHUB_USERNAME>/ed-deterioration-forecast.git"   # <- edit once
BRANCH = "main"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{tok}@{REPO}"
except Exception:
    url = f"https://{REPO}"
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/repo[train]"], check=True)
os.chdir("/kaggle/working/repo")

In [ ]:
# --- Paths: keep RAW identical in every notebook (it is part of the cache key) ---
RAW  = "/kaggle/input/mc-med"                 # <- your PRIVATE Kaggle dataset with MC-MED CSVs
WORK = "/kaggle/input/<notebook-01-output-name>/work"
RUNS = "/kaggle/working/runs"
COMMON = [f"paths.raw_dir={RAW}", f"paths.work_dir={WORK}", f"paths.runs_dir={RUNS}"]
ARGS = " ".join(COMMON)


In [ ]:
# Baselines first (seconds), then the learned models. Add e.g. "train.tracker=wandb" to log to W&B
# (put WANDB_API_KEY in Kaggle Secrets and export it as an env var first).
for model in ["persistence", "news2_rule", "xgb", "gru", "tcn"]:
    !edews run --stages evaluate {ARGS} model={model}

In [ ]:
!edews compare {ARGS}

### Pick the model on **validation** results (rule fixed in advance), then export it for the Raspberry Pi
The bundle folder (model + normaliser + edge config + replay stream) is what you copy to the Pi.

In [ ]:
MODEL = "tcn"     # <- the model your selection rule picked
!edews run --stages export {ARGS} model={MODEL}
!ls -la {RUNS}/{MODEL}_*/export/*/

In [ ]:
# Sanity check the bundle here before copying it to the Pi
import glob
bundle = sorted(glob.glob(f"{RUNS}/{MODEL}_*/export/*/"))[-1]
!python -m edews.edge.replay --bundle {bundle}

### Ablations (each is one override; data stages stay cached unless the override touches them)
```
labels.horizon_end=30        # shorter horizon            -> relabel + reindex + retrain
labels.horizon_end=120
windows.length_min=30        # shorter window             -> reindex + retrain
train.loss_weight_current=0  # future-only (no multi-task) -> retrain only
seed=1 / seed=2              # repeat for mean ± sd
```